In [ ]:
import jsonlines
import os
import json
from collections import defaultdict
import re
import random
import jieba
random.seed(2025)

In [ ]:
gold_file = 'xx'
pred_file = 'xx'


In [ ]:
label_list_1 = ["预后", "部位", "药物", "社会学", "症状", "疾病", "流行病学", "检查", "手术治疗", "其他治疗", "其他"]
label_list_2 = ["临床表现", "传播途径", "侵及周围组织转移的症状", "内窥镜检查", "化疗", "发病年龄", "发病性别倾向", "发病机制", "发病率", "发病部位", "同义词", "外侵部位", "多发地区", "多发季节", "多发群体", "实验室检查", "就诊科室", "并发症", "影像学检查", "手术治疗", "放射治疗", "死亡率", "治疗后症状", "病史", "病因", "病理分型", "病理生理", "相关（导致）", "相关（症状）", "相关（转化）", "筛查", "组织学检查", "药物治疗", "转移部位", "辅助检查", "辅助治疗", "遗传因素", "鉴别诊断", "阶段", "预后状况", "预后生存率", "预防", "风险评估因素", "高危因素"]

In [ ]:
with jsonlines.open(gold_file, 'r') as f:
    gold_datas = [data for data in f]
with jsonlines.open(pred_file, 'r') as f:
    pred_datas = [data for data in f]
pred_dict = {}
for data in pred_datas:
    try:
        text = re.findall(r'文本：“(.*?)”\n', data['prompt'])[0]
    except:
        text = re.findall(r'给定信息：\n文本：(.*?)\n', data['prompt'])[0]
    pred_dict[text] = data

In [ ]:
data['prompt']

In [ ]:
len(gold_datas), len(pred_datas)

In [ ]:
import re

def get_spoes(text, min_fields=1, max_fields=5):
    # 匹配全角/半角括号里的片段（不跨越右括号）
    pattern = re.compile(r'[（(]((?:(?![）)]).)*?)[）)]')
    results = []
    for m in pattern.finditer(text):
        inner = m.group(1)
        parts = [p.strip() for p in re.split(r'\|\|', inner)]
        if min_fields <= len(parts) <= max_fields:
            results.append(tuple(parts))
    return results

# 用法
# spoes = parse_tuples(text, min_fields=2, max_fields=5)


In [ ]:
def get_metric(pred_spoes, gold_spoes):
    try:
        pred_tuples = set(pred_spoes)
        gold_tuples = set(gold_spoes)
        corr_tuples = pred_tuples & gold_tuples
        # print('pred_tuples:{}'.format(pred_tuples))
        # print('gold_tuples:{}'.format(gold_tuples))
        pre = len(corr_tuples) / len(pred_tuples)
        rec = len(corr_tuples) / len(gold_tuples)
        f1 = 2*pre*rec/(pre+rec)
    except:
        pre = 0
        rec = 0
        f1 = 0
    return pre, rec, f1

In [ ]:
def transfer_spoes_to_str(spoes):
    s = ''
    for item in spoes:
        s += '（{}）'.format('||'.join(item))
    s = '```\n{}\n```'.format(s.strip())
    return s

In [ ]:
def random_phrase(text, lang="auto", max_len=5):
    """
    从文本中随机获取一个词组
    :param text: 输入文本
    :param lang: 'zh' 表示中文，'en' 表示英文，'auto' 自动判断
    :param max_len: 最大词组长度
    :return: 随机词组字符串
    """
    # 自动判断：含中文字符则认为是中文
    if lang == "auto":
        if re.search(r'[\u4e00-\u9fff]', text):
            lang = "zh"
        else:
            lang = "en"

    if lang == "zh":
        words = list(jieba.cut(text))
    else:  # 英文
        words = text.split()

    if not words:
        return ""

    # 随机选起点与长度
    start = random.randint(0, len(words) - 1)
    length = random.randint(1, min(max_len, len(words) - start))
    phrase = "".join(words[start:start+length]) if lang == "zh" else " ".join(words[start:start+length])
    return phrase

In [ ]:
def get_fake_data(text, spoes):
    processed_spoes = []
    if len(spoes) >= 2:
        del_nums = random.randint(1,len(spoes)-1)
        del_spoes = random.sample(spoes, del_nums)
        for item in spoes:
            if item in del_spoes:
                continue
            processed_spoes.append(item)
        return processed_spoes
    spo_len = len(spoes[0])
    now_spo = spoes[0]
    select_index = random.randint(0,spo_len-1)
    select_item = now_spo[select_index]
    if select_item in label_list_1:
        now_item = select_item
        new_item = select_item
        while now_item == new_item:
            new_item = random.choice(label_list_1)
    elif select_item in label_list_2:
        now_item = select_item
        new_item = select_item
        while now_item == new_item:
            new_item = random.choice(label_list_2)
    else:
        now_item = select_item
        new_item = select_item
        while now_item == new_item:
            new_item = random_phrase(text)
    print('now_item:{}\tnew_item:{}'.format(now_item, new_item))
    res_item = []
    for i in range(spo_len):
        if i != select_index:
            res_item.append(now_spo[i])
        else:
            res_item.append(new_item)
    return [tuple(res_item)]

In [ ]:
processed_datas = []
# chosen, rejected, score_chosen, score_rejected
# chosen = 标答
# rejected = 预测
# score_chosen = 1
# score_rejected = f1(pred)

for index,data in enumerate(gold_datas):
    try:
        text = re.findall(r'给定文本：“(.*?)”\n', data['instruction'])[0]
    except:
        text = re.findall(r'给定信息：\n文本：(.*?)\n', data['instruction'])[0]
    if text in pred_dict.keys():
        pred_data = pred_dict[text]
    else:
        pred_data = ''
    if 'pred' in data.keys() and 'gold' in data.keys():
        if '||' in data['pred'][0]:
            pred_spoes = []
            gold_spoes = []
            for item in data['pred']:
                pred_spoes.append(tuple(item[1:-1].split('||')))
            for item in data['gold']:
                gold_spoes.append(tuple(item[1:-1].split('||')))
        else:
            pred_spoes = [(data['pred'][0],data['pred'][1],data['pred'][2],data['pred'][3],data['pred'][4])]
            gold_spoes = [(data['gold'][0],data['gold'][1],data['gold'][2],data['gold'][3],data['gold'][4])]
        chosen_str = transfer_spoes_to_str(gold_spoes)
        rejected_str = transfer_spoes_to_str(pred_spoes)
        pre, rec, f1 = get_metric(pred_spoes, gold_spoes)
        assert f1 < 1.
        score_chosen = 1.
        score_rejected = f1
        chosen = []
        chosen.append({"role":"user", "content":data['instruction']})
        chosen.append({"role":"assistant", "content":chosen_str})
        rejected = []
        rejected.append({"role":"user", "content":data['instruction']})
        rejected.append({"role":"assistant", "content":rejected_str})
        processed_datas.append({
            'chosen':chosen,
            'rejected':rejected,
            'score_chosen':score_chosen,
            'score_rejected':score_rejected,
            'type': 'has_pred_gold_key'
        })
        continue
    # 如果有预测答案
    if pred_data != '':
        # 判断下预测结果是不是完全正确
        pred_spoes = get_spoes(pred_data['output'])
        gold_spoes = get_spoes(data['output'])
        pre, rec, f1 = get_metric(pred_spoes, gold_spoes)
        if f1 == 1.:
            # 如果完全正确
            fake_spoes = get_fake_data(text, gold_spoes)
            print('gold_spoes:{}'.format(gold_spoes))
            print('fake_spoes:{}'.format(fake_spoes))
            chosen_str = transfer_spoes_to_str(gold_spoes)
            rejected_str = transfer_spoes_to_str(fake_spoes)
            chosen = []
            chosen.append({"role":"user", "content":data['instruction']})
            chosen.append({"role":"assistant", "content":chosen_str})
            rejected = []
            rejected.append({"role":"user", "content":data['instruction']})
            rejected.append({"role":"assistant", "content":rejected_str})
            pre, rec, f1 = get_metric(fake_spoes, gold_spoes)
            assert f1 < 1.
            score_chosen = 1.
            score_rejected = f1
            processed_datas.append({
                'chosen':chosen,
                'rejected':rejected,
                'score_chosen':score_chosen,
                'score_rejected':score_rejected,
                'type': 'has_pred_data_f1_eq_1'
            })
        else:
            chosen_str = transfer_spoes_to_str(gold_spoes)
            rejected_str = transfer_spoes_to_str(pred_spoes)
            score_chosen = 1.
            score_rejected = f1
            chosen = []
            chosen.append({"role":"user", "content":data['instruction']})
            chosen.append({"role":"assistant", "content":chosen_str})
            rejected = []
            rejected.append({"role":"user", "content":data['instruction']})
            rejected.append({"role":"assistant", "content":rejected_str})
            processed_datas.append({
                'chosen':chosen,
                'rejected':rejected,
                'score_chosen':score_chosen,
                'score_rejected':score_rejected,
                'type': 'has_pred_data_f1_below_1'
            })
    else:
        gold_spoes = get_spoes(data['output'])
        fake_spoes = get_fake_data(text, gold_spoes)
        print('gold_spoes:{}'.format(gold_spoes))
        print('fake_spoes:{}'.format(fake_spoes))
        chosen_str = transfer_spoes_to_str(gold_spoes)
        rejected_str = transfer_spoes_to_str(fake_spoes)
        pre, rec, f1 = get_metric(fake_spoes, gold_spoes)
        assert f1 < 1.
        score_chosen = 1.
        score_rejected = f1
        chosen = []
        chosen.append({"role":"user", "content":data['instruction']})
        chosen.append({"role":"assistant", "content":chosen_str})
        rejected = []
        rejected.append({"role":"user", "content":data['instruction']})
        rejected.append({"role":"assistant", "content":rejected_str})
        processed_datas.append({
            'chosen':chosen,
            'rejected':rejected,
            'score_chosen':score_chosen,
            'score_rejected':score_rejected,
            'type': 'no_pred_data'
            
        })
    
    # if data['instruction'] in gold_dict.keys():
    #     gold_data = 

In [ ]:
print(data['instruction'])

In [ ]:
data

In [ ]:
with jsonlines.open('{}.jsonl'.format(dataset_name), 'w') as f:
    for data in processed_datas:
        f.write(data)